# Session 9b — What the model actually sees
**Tokens · cost · the context window · embeddings**

Run every cell top to bottom (Kernel → Restart & Run All). Each section ends with a question in *italics* — answer it in the reflection at the end.

Before starting: `git pull` and `uv sync` in the course repo, and open this notebook with `uv run jupyter lab` (or in VS Code with the project's `.venv` selected).

## 1. Setup — two providers, one client library
A **provider** is *where* the model runs; a **model** is *which brain* answers. The same client library talks to both — only `base_url` changes. We use Groq (from `.env`) for chat and Ollama (on your laptop) for embeddings.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
from tokens_utils import *          # count_tokens, show_tokens, cost_inr, chat_client, ollama_client, usage_for, load_points, WORDS
import os
MODEL = os.getenv("MODEL")
groq = chat_client()                # Groq — from .env
ollama = ollama_client()            # local — http://localhost:11434/v1
print("chat model:", MODEL)

## 2. What a token is
A **tokenizer** is a fixed dictionary that splits text into pieces the model can read. The pieces are **tokens** — roughly ¾ of an English word each. The dictionary was built mostly from English and code, so other scripts cost more.

Our default model (`openai/gpt-oss-20b`) uses the `o200k_harmony` encoding, which `tiktoken` ships — so for this model the count is **exact**. For other models it is an estimate; the model's own `usage` field is the truth after you send (section 3).

In [ ]:
TEXTS = {
    "english": "Can I get a personal loan of 8 lakh rupees for five years?",
    "telugu":  "నాకు ఐదు సంవత్సరాలకు 8 లక్షల రూపాయల వ్యక్తిగత రుణం లభిస్తుందా?",
    "python":  "def emi(p, r, n):\n    r = r/12/100\n    return p*r*(1+r)**n/((1+r)**n-1)",
    "json":    '{"pan": "ABCDE1234F", "amount": 800000, "tenure_months": 60}',
}
print(f"{'text':8} {'chars':>6} {'tokens':>7} {'chars/token':>12}")
for name, t in TEXTS.items():
    n = count_tokens(t)
    print(f"{name:8} {len(t):6d} {n:7d} {len(t)/n:12.1f}")

In [ ]:
# See the pieces. Where does the tokenizer cut?
for name in ("english", "telugu"):
    print(name, "→", show_tokens(TEXTS[name]), "\n")

*Question 1 — The Telugu sentence means the same as the English one. Why does it cost more tokens, and what does that mean for a bank whose customers write in Telugu?*

## 3. Check against reality — the `usage` field
Send the same texts to a real model and read `usage.prompt_tokens`. The model adds a few *template tokens* around your message (role markers), so expect the real number to be slightly higher than the raw count. If Ollama is running, send them there too — a **different model, a different tokenizer, a different count**.

In [ ]:
def usage(client, model, text):
    return usage_for(client, model, text)[0]

overhead_groq = usage(groq, MODEL, "hi") - count_tokens("hi")
print(f"template wrapper on Groq/{MODEL}: {overhead_groq} tokens you never typed\n")
print(f"{'text':8} {'tiktoken':>9} {'groq usage':>11} {'minus wrapper':>14} {'ollama usage':>13}")
for name, t in TEXTS.items():
    est = count_tokens(t)
    g = usage(groq, MODEL, t)
    try:
        o = str(usage(ollama, "llama3.2:3b", t))
    except Exception:
        o = "(no ollama)"
    print(f"{name:8} {est:9d} {g:11d} {g - overhead_groq:14d} {o:>13}")

*Question 2 — Tokens are a property of the model, not the text. Which row shows this most clearly?*

## 4. What it costs
Providers charge per token, in and out, per million. The price cards below are **indicative** — edit them from the provider's pricing page. We convert to ₹ and scale to 1,000 calls, because a chatbot does not answer once.

In [ ]:
for card in PRICE_CARDS:
    print(f"\n{card}")
    for name, t in TEXTS.items():
        p_in = count_tokens(t); p_out = 120          # assume a 120-token answer
        print(f"  {name:8} ₹{cost_inr(p_in, p_out, card):8.4f} per call   ₹{cost_inr(p_in, p_out, card, calls=1000):8.2f} per 1,000 calls")

*Question 3 — Output tokens cost more than input tokens on every card. What does that suggest about how you should ask a model to answer?*

## 5. The context window as a budget
A model sees a fixed number of tokens at once — its **context window**. Everything must fit: your system prompt, the whole conversation so far, any documents you paste in, any tool results. Here we simulate a growing conversation and watch the count climb. Where the line crosses the limit, something has to be dropped — that is Session 19's problem (memory windows) and Session 38's (context engineering).

In [ ]:
import matplotlib.pyplot as plt
system = "You are a concise assistant for a retail bank."
history = [{"role": "system", "content": system}]
totals = []
for turn in range(1, 21):
    history.append({"role": "user", "content": f"Turn {turn}: {TEXTS['english']}"})
    history.append({"role": "assistant", "content": "Certainly. " + "Here is a detailed answer about your loan. " * 6})
    totals.append(sum(count_tokens(m["content"]) for m in history))
LIMIT = 8192   # a small window, for the picture; real models are far larger — but so are real conversations with documents
plt.plot(range(1, 21), totals, marker="o"); plt.axhline(LIMIT, color="red", linestyle="--", label=f"limit {LIMIT}")
plt.xlabel("turn"); plt.ylabel("tokens in context"); plt.title("The conversation grows every turn"); plt.legend(); plt.show()
print("tokens at turn 20:", totals[-1])

*Question 4 — At turn 21, what is in the window and what has to go? Who decides?*

## 6. Embeddings — meaning as coordinates
An **embedding model** does not answer. It turns text into a **vector** — a list of numbers (768 here) — such that similar meanings land close together. We embed ten words with a local model, squash 768 dimensions to 2 with PCA (plain NumPy), and draw them. If Ollama is not running, the notebook loads a cached copy made with the same model.

Where this comes back: **Module 4** (vector databases), **Module 5** (RAG), **Session 39** (agentic RAG).

In [ ]:
pts, source = load_points(WORDS)
print("source:", source)
plt.figure(figsize=(7, 5))
plt.scatter(pts[:, 0], pts[:, 1])
for w, (x, y) in zip(WORDS, pts):
    plt.annotate(w, (x, y), xytext=(4, 4), textcoords="offset points")
plt.title("Ten words, two directions — nearby means similar"); plt.show()

*Question 5 — Which words cluster? Which word sits alone, and why? What can an embedding model **not** do?*

## 7. Reflection
Answer the five questions above here, one or two lines each.

1. 
2. 
3. 
4. 
5. 

Then run the checkpoint from the terminal: `uv run pytest tests/test_s09.py`